In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import
train_test

In [ ]:
pip install requests beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup

print("Requests and BeautifulSoup are ready!")

Requests and BeautifulSoup are ready!


In [ ]:

import requests
from bs4 import BeautifulSoup

# Website URL
url = "https://books.toscrape.com/"

# Send a request to the website
response = requests.get(url, timeout=15)

# Check whether the request was successful
print("Status Code:", response.status_code)

# Convert HTML into a readable structure
soup = BeautifulSoup(response.text, "html.parser")

# Print the webpage title
print("Website Title:", soup.title.text)

# Print the first 500 characters of HTML
print("\nHTML Preview:")
print(response.text[:500])


Status Code: 200
Website Title: 
    All products | Books to Scrape - Sandbox


HTML Preview:
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


In [ ]:

# Find all book listings on the page
books = soup.select("article.product_pod")

print("Total books on this page:", len(books))

# Extract the first book's information
first_book = books[0]

title = first_book.select_one("h3 a")["title"]
price = first_book.select_one(".price_color").text
availability = first_book.select_one(".availability").get_text(strip=True)

print("\nFirst Book Details")
print("Title:", title)
print("Price:", price)
print("Availability:", availability)


Total books on this page: 20

First Book Details
Title: A Light in the Attic
Price: Â£51.77
Availability: In stock


In [ ]:

# Find all book cards on the current page
books = soup.select("article.product_pod")

# Create an empty list to store book details
book_data = []

for book in books:
    title = book.select_one("h3 a")["title"]
    price = book.select_one(".price_color").get_text(strip=True)
    availability = book.select_one(".availability").get_text(strip=True)

    rating_tag = book.select_one("p.star-rating")
    rating = rating_tag["class"][-1] if rating_tag else "Unknown"

    book_data.append({
        "title": title,
        "price": price,
        "availability": availability,
        "rating": rating
    })

print("Total books extracted:", len(book_data))

# Display the extracted records
for item in book_data:
    print(item)


Total books extracted: 20
{'title': 'A Light in the Attic', 'price': 'Â£51.77', 'availability': 'In stock', 'rating': 'Three'}
{'title': 'Tipping the Velvet', 'price': 'Â£53.74', 'availability': 'In stock', 'rating': 'One'}
{'title': 'Soumission', 'price': 'Â£50.10', 'availability': 'In stock', 'rating': 'One'}
{'title': 'Sharp Objects', 'price': 'Â£47.82', 'availability': 'In stock', 'rating': 'Four'}
{'title': 'Sapiens: A Brief History of Humankind', 'price': 'Â£54.23', 'availability': 'In stock', 'rating': 'Five'}
{'title': 'The Requiem Red', 'price': 'Â£22.65', 'availability': 'In stock', 'rating': 'One'}
{'title': 'The Dirty Little Secrets of Getting Your Dream Job', 'price': 'Â£33.34', 'availability': 'In stock', 'rating': 'Four'}
{'title': 'The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull', 'price': 'Â£17.93', 'availability': 'In stock', 'rating': 'Three'}
{'title': 'The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 

In [ ]:

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

base_url = "https://books.toscrape.com/"
current_url = base_url

book_data = []
max_records = 100

while current_url and len(book_data) < max_records:
    print("Scraping:", current_url)

    response = requests.get(current_url, timeout=15)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Extract books from the current page
    for book in soup.select("article.product_pod"):
        title_tag = book.select_one("h3 a")
        price_tag = book.select_one(".price_color")
        availability_tag = book.select_one(".availability")
        rating_tag = book.select_one("p.star-rating")

        if not title_tag or not price_tag:
            continue

        book_data.append({
            "title": title_tag.get("title", title_tag.get_text(strip=True)),
            "price": price_tag.get_text(strip=True),
            "availability": (
                availability_tag.get_text(" ", strip=True)
                if availability_tag else "Unknown"
            ),
            "rating": (
                rating_tag["class"][-1]
                if rating_tag else "Unknown"
            ),
            "page_url": current_url
        })

        if len(book_data) >= max_records:
            break

    # Find the link to the next page
    next_link = soup.select_one("li.next a")

    if next_link:
        current_url = urljoin(current_url, next_link["href"])
    else:
        current_url = None

    time.sleep(1)

print("\nTotal records collected:", len(book_data))


Scraping: https://books.toscrape.com/
Scraping: https://books.toscrape.com/catalogue/page-2.html
Scraping: https://books.toscrape.com/catalogue/page-3.html
Scraping: https://books.toscrape.com/catalogue/page-4.html
Scraping: https://books.toscrape.com/catalogue/page-5.html

Total records collected: 100


In [ ]:

import csv

with open("books.csv", "w", newline="", encoding="utf-8-sig") as file:
    fieldnames = book_data[0].keys()

    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(book_data)

print("CSV file created successfully!")



CSV file created successfully!


In [ ]:

import json

with open("books.json", "w", encoding="utf-8") as file:
    json.dump(book_data, file, indent=4, ensure_ascii=False)

print("JSON file created successfully!")


JSON file created successfully!


In [ ]:

from google.colab import files

files.download("books.csv")
files.download("books.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import csv

# Fix the currency symbol in every record
for book in book_data:
    book["price"] = book["price"].replace("Â£", "£")

# Save the corrected CSV
with open("books.csv", "w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(file, fieldnames=book_data[0].keys())
    writer.writeheader()
    writer.writerows(book_data)

print("Price formatting fixed!")
print("Total records:", len(book_data))
print("Example price:", book_data[0]["price"])


Price formatting fixed!
Total records: 100
Example price: £51.77


In [ ]:
from google.colab import files
files.download("books.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>